# DroneDetect V2 - RF-UAVNet Training

**Reference:** Huynh-The et al. (2022), "RF-UAVNet: High-Performance Convolutional Network for RF-Based Drone Surveillance Systems", *IEEE Access*, Vol. 10. [DOI](https://doi.org/10.1109/ACCESS.2022.3173181)

RF-UAVNet is a lightweight 1D CNN (4,615 parameters) designed for RF-based drone classification from raw IQ signals. The architecture chains an R-Unit (Conv1d 2->64), four G-Units with grouped convolutions and skip connections, a multi-scale global average pooling layer, and a fully connected classifier (320 -> 7 classes).

This notebook trains and evaluates RF-UAVNet on the DroneDetect V2 dataset. Input signals are downsampled ~120x (1.2M to 10K samples per 20 ms segment), feeding raw IQ data (2 channels: real + imaginary) directly into the network without frequency-domain transformation.

## 0. Colab Setup
Run this cell only on Google Colab. Skipped automatically when running locally.

In [1]:
import os
import subprocess

COLAB = "COLAB_RELEASE_TAG" in os.environ

if COLAB:
    subprocess.check_call(["pip", "install", "-q", "boto3"])
    from google.colab import userdata
    import boto3

    s3 = boto3.client(
        "s3",
        endpoint_url="https://s3.fr-par.scw.cloud",
        region_name="fr-par",
        aws_access_key_id=userdata.get("SCW_ACCESS_KEY"),
        aws_secret_access_key=userdata.get("SCW_SECRET_KEY"),
    )
    ARTIFACT_BUCKET = "mldrone-artefacts"

    os.makedirs("../data/features", exist_ok=True)
    os.makedirs("../data/models", exist_ok=True)

    downloads = {
        "features/iq_features.npz": "../data/features/iq_features.npz",
        "split/split_indices.npz": "../data/split_indices.npz",
    }
    for s3_key, local_path in downloads.items():
        if not os.path.exists(local_path):
            print(f"Downloading {s3_key}...")
            s3.download_file(ARTIFACT_BUCKET, s3_key, local_path)
            print(f"Done: {local_path}")

Done: ../data/features/iq_features.npz
Done: ../data/split_indices.npz


## 1. Imports and Configuration

In [2]:
import copy
import gc
import logging
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    f1_score, precision_recall_fscore_support,
)
from tqdm import tqdm

try:
    from dronedetect.splitting import create_stratified_split, load_split, verify_split_balance
except ImportError:
    from sklearn.model_selection import StratifiedGroupKFold

    def create_stratified_split(
        y: np.ndarray,
        file_ids: np.ndarray,
        train_size: float = 0.70,
        val_size: float = 0.15,
        test_size: float = 0.15,
        random_state: int = 42,
        save_path: str | Path | None = None,
    ) -> dict:
        """Two-stage stratified group split into train/val/test partitions.

        Stage 1: split off test set using StratifiedGroupKFold.
        Stage 2: split remaining trainval into train and val.

        Stratification is by drone label (y). Grouping is by file_ids to prevent
        segments from the same recording appearing in different partitions.
        """
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)

        total = train_size + val_size + test_size
        if not np.isclose(total, 1.0):
            raise ValueError(f"Split fractions must sum to 1.0, got {total:.4f}")

        n_splits_test = round(1 / test_size)
        sgkf_test = StratifiedGroupKFold(
            n_splits=n_splits_test, shuffle=True, random_state=random_state
        )
        all_indices = np.arange(len(y))

        trainval_idx, test_idx = next(sgkf_test.split(all_indices, y, groups=file_ids))

        relative_val = val_size / (train_size + val_size)
        n_splits_val = round(1 / relative_val)
        sgkf_val = StratifiedGroupKFold(
            n_splits=n_splits_val, shuffle=True, random_state=random_state + 1
        )

        y_trainval = y[trainval_idx]
        file_ids_trainval = file_ids[trainval_idx]

        train_local, val_local = next(
            sgkf_val.split(
                np.arange(len(trainval_idx)), y_trainval, groups=file_ids_trainval
            )
        )
        train_idx = trainval_idx[train_local]
        val_idx = trainval_idx[val_local]

        train_files = set(file_ids[train_idx])
        val_files = set(file_ids[val_idx])
        test_files = set(file_ids[test_idx])

        overlap_tv = train_files & val_files
        overlap_tt = train_files & test_files
        overlap_vt = val_files & test_files

        if overlap_tv or overlap_tt or overlap_vt:
            msg = "Data leakage detected! File overlap between partitions:"
            if overlap_tv:
                msg += f"\n  train-val: {overlap_tv}"
            if overlap_tt:
                msg += f"\n  train-test: {overlap_tt}"
            if overlap_vt:
                msg += f"\n  val-test: {overlap_vt}"
            raise ValueError(msg)

        logger.info("Split created with zero file overlap (leakage-free)")
        logger.info(
            "Samples  -> train=%d, val=%d, test=%d (total=%d)",
            len(train_idx),
            len(val_idx),
            len(test_idx),
            len(y),
        )
        logger.info(
            "Files    -> train=%d, val=%d, test=%d",
            len(train_files),
            len(val_files),
            len(test_files),
        )

        split_metadata = {
            "train_samples": len(train_idx),
            "val_samples": len(val_idx),
            "test_samples": len(test_idx),
            "train_files": len(train_files),
            "val_files": len(val_files),
            "test_files": len(test_files),
            "random_state": random_state,
        }

        result = {
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "split_metadata": split_metadata,
        }

        if save_path is not None:
            save_path = Path(save_path)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez(
                save_path,
                train_idx=train_idx,
                val_idx=val_idx,
                test_idx=test_idx,
            )
            logger.info("Split indices saved to %s", save_path)

        return result

    def verify_split_balance(
        file_ids: np.ndarray,
        y: np.ndarray,
        split_indices: dict,
        conditions: np.ndarray | None = None,
    ) -> None:
        """Log class and condition distributions per partition for diagnostics."""
        y = np.asarray(y)
        file_ids = np.asarray(file_ids)
        if conditions is not None:
            conditions = np.asarray(conditions)

        partitions = {
            "train": split_indices["train_idx"],
            "val": split_indices["val_idx"],
            "test": split_indices["test_idx"],
        }

        all_classes = np.unique(y)

        for name, idx in partitions.items():
            y_part = y[idx]
            classes, counts = np.unique(y_part, return_counts=True)
            dist_str = ", ".join(f"{c}={n}" for c, n in zip(classes, counts))
            logger.info("[%s] Class distribution: %s", name, dist_str)

        if conditions is not None:
            all_conditions = np.unique(conditions)

            for name, idx in partitions.items():
                cond_part = conditions[idx]
                conds, counts = np.unique(cond_part, return_counts=True)
                dist_str = ", ".join(f"{c}={n}" for c, n in zip(conds, counts))
                logger.info("[%s] Condition distribution: %s", name, dist_str)

                y_part = y[idx]
                for drone in all_classes:
                    drone_mask = y_part == drone
                    drone_conditions = set(np.unique(cond_part[drone_mask]))
                    missing = set(all_conditions) - drone_conditions
                    if missing:
                        logger.warning(
                            "[%s] Drone '%s' missing conditions: %s",
                            name,
                            drone,
                            missing,
                        )

    def load_split(path: str | Path) -> dict:
        """Load previously saved split indices from a .npz file."""
        data = np.load(path)
        return {
            "train_idx": data["train_idx"],
            "val_idx": data["val_idx"],
            "test_idx": data["test_idx"],
        }

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Using device: %s", device)

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

NOTEBOOK_NAME = "training_rfuavnet"
FIGURES_DIR = Path("figures") / NOTEBOOK_NAME


def save_figure(fig) -> None:
    """Save plotly figure to PNG file using the figure's title as filename."""
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    title = fig.layout.title.text if fig.layout.title.text else "untitled"
    filename = re.sub(r'[^\w\s-]', '', title).strip()
    filename = re.sub(r'[\s-]+', '_', filename)
    filepath = FIGURES_DIR / f"{filename}.png"
    try:
        fig.write_image(str(filepath), width=1200, height=800)
        logger.info("Saved: %s", filepath)
    except Exception as e:
        logger.warning("Could not save figure (kaleido required): %s", e)

In [3]:
FEATURES_DIR = Path("../data/features")
SPLIT_PATH = Path("../data/split_indices.npz")

CONFIG = {
    'models_dir': '../data/models/',
    'test_data_dir': '../data/test_samples/',

    # Training parameters (aligned with Huynh-The et al. 2022, IEEE Access)
    'batch_size': 512,
    'max_epochs': 120,
    'patience': 10,
    'learning_rate': 0.001,
    'weight_decay': 1e-4,
    'scheduler_factor': 0.5,
    # ES patience >= 2x scheduler patience per design choice
    'scheduler_patience': 4,

    'device': device,
}

logger.info("Configuration: %s", CONFIG)

### Hyperparameters

- **Optimizer**: Adam (lr=0.001, weight_decay=1e-4)
- **Batch size**: 512
- **Max epochs**: 120 with early stopping (patience=10)
- **LR scheduler**: ReduceLROnPlateau (factor=0.5, patience=4)

## 2. RF-UAV-Net Model Definition

In [4]:
class RFUAVNet(nn.Module):
    """RF-UAVNet: 1D CNN for RF-based drone classification.

    Architecture from Huynh-The et al. (2022), IEEE Access.
    Processes raw IQ signals through R-Unit, G-Units with skip connections,
    and multi-scale global average pooling.
    """

    def __init__(self, num_classes: int):
        super().__init__()

        self.conv_r = nn.Conv1d(2, 64, kernel_size=5, stride=5)
        self.bn_r = nn.BatchNorm1d(64)
        self.elu_r = nn.ELU()

        self.g_convs = nn.ModuleList([
            nn.Conv1d(64, 64, kernel_size=3, stride=2, groups=8)
            for _ in range(4)
        ])
        self.g_bns = nn.ModuleList([nn.BatchNorm1d(64) for _ in range(4)])
        self.g_elus = nn.ModuleList([nn.ELU() for _ in range(4)])

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)

        self.gap1000 = nn.AvgPool1d(1000)
        self.gap500 = nn.AvgPool1d(500)
        self.gap250 = nn.AvgPool1d(250)
        self.gap125 = nn.AvgPool1d(125)

        self.fc = nn.Linear(320, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass: (batch_size, 2, 10000) -> (batch_size, num_classes)."""
        x = self.elu_r(self.bn_r(self.conv_r(x)))

        g_outputs = []
        for i in range(4):
            # Left-pad by 1 to maintain temporal dimension after k=3 conv
            g_out = self.g_elus[i](self.g_bns[i](self.g_convs[i](F.pad(x, (1, 0)))))
            g_outputs.append(g_out)
            x = g_out + self.pool(x)

        gaps = [
            self.gap1000(g_outputs[0]),
            self.gap500(g_outputs[1]),
            self.gap250(g_outputs[2]),
            self.gap125(g_outputs[3]),
            self.gap125(x),
        ]

        x = torch.cat(gaps, dim=1).flatten(start_dim=1)
        return self.fc(x)

## 3. Load IQ Features

In [5]:
data = np.load(FEATURES_DIR / "iq_features.npz", mmap_mode="r")

X = data['X']
y_drone = data['y_drone']
y_interference = data['y_interference']
y_state = data['y_state']
file_ids = data['file_ids']

drone_classes = np.array(data['drone_classes'])
interference_classes = np.array(data['interference_classes'])
state_classes = np.array(data['state_classes'])

logger.info("IQ data shape: %s", X.shape)
logger.info("Drone labels shape: %s", y_drone.shape)
logger.info("File IDs: %d unique files", len(np.unique(file_ids)))
logger.info("Drone classes: %s", drone_classes)

## 4. Train/Val/Test Split

70/15/15 file-level stratified split to prevent data leakage. Shared across all notebooks via `splitting.py`.

In [6]:
if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
    logger.info("Loaded existing split from %s", SPLIT_PATH)
else:
    split = create_stratified_split(y_drone, file_ids, save_path=SPLIT_PATH)
    logger.info("Created and saved new split to %s", SPLIT_PATH)

train_idx = split["train_idx"]
val_idx = split["val_idx"]
test_idx = split["test_idx"]

verify_split_balance(file_ids, y_drone, split, conditions=y_interference)

y_train = y_drone[train_idx]
y_val = y_drone[val_idx]
y_test = y_drone[test_idx]

logger.info("Train: %d | Val: %d | Test: %d samples", len(train_idx), len(val_idx), len(test_idx))

# Save bulk test data for post-training evaluation
test_data_dir = Path(CONFIG['test_data_dir'])
test_data_dir.mkdir(parents=True, exist_ok=True)

X_train = X[train_idx]
X_val = X[val_idx]
X_test = X[test_idx]

test_data_path = test_data_dir / "rfuavnet_test_data.npz"
np.savez(
    test_data_path,
    X_test=X_test,
    y_test=y_test,
    y_interference_test=y_interference[test_idx],
    y_state_test=y_state[test_idx],
    test_idx=test_idx,
    file_ids_test=file_ids[test_idx],
    drone_classes=drone_classes,
    interference_classes=interference_classes,
    state_classes=state_classes,
)
logger.info("Test data saved to %s", test_data_path)

## 5. Prepare PyTorch Datasets

In [7]:
X_train_t = torch.from_numpy(X_train).float()
y_train_t = torch.from_numpy(y_train).long()
del X_train, y_train
gc.collect()

X_val_t = torch.from_numpy(X_val).float()
y_val_t = torch.from_numpy(y_val).long()
del X_val, y_val
gc.collect()

X_test_t = torch.from_numpy(X_test).float()
y_test_t = torch.from_numpy(y_test).long()
del X_test, y_test, X, data
gc.collect()

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

logger.info("Train batches: %d | Val batches: %d | Test batches: %d",
            len(train_loader), len(val_loader), len(test_loader))

## 6. Training Function

In [8]:
def train_model(model, train_loader, val_loader, test_loader, config):
    """Train RF-UAVNet with early stopping and LR scheduling on val loss.

    Uses Adam optimizer (aligned with Huynh-The 2022 paper and IQTLabs
    reference repo) and ReduceLROnPlateau on validation loss.

    Returns the best model (lowest val_loss) restored after training.
    """
    model = model.to(config['device'])
    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay'],
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=config['scheduler_factor'],
        patience=config['scheduler_patience'],
    )

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'test_loss': [], 'test_acc': [],
        'lr': [],
    }

    best_val_loss = float('inf')
    epochs_without_improvement = 0
    best_model_state = None
    max_epochs = config['max_epochs']
    patience = config['patience']

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for batch_x, batch_y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}"):
            batch_x = batch_x.to(config['device'])
            batch_y = batch_y.to(config['device'])

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += batch_y.size(0)
            train_correct += (predicted == batch_y).sum().item()

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(config['device'])
                batch_y = batch_y.to(config['device'])

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += batch_y.size(0)
                val_correct += (predicted == batch_y).sum().item()

        test_loss = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(config['device'])
                batch_y = batch_y.to(config['device'])

                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)

                test_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                test_total += batch_y.size(0)
                test_correct += (predicted == batch_y).sum().item()

        avg_val_loss = val_loss / len(val_loader)
        avg_test_loss = test_loss / len(test_loader)
        scheduler.step(avg_val_loss)

        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(train_correct / train_total)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_correct / val_total)
        history['test_loss'].append(avg_test_loss)
        history['test_acc'].append(test_correct / test_total)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        logger.info(
            "Epoch %d/%d: Train Loss=%.4f, Train Acc=%.4f, "
            "Val Loss=%.4f, Val Acc=%.4f, Test Loss=%.4f, Test Acc=%.4f, LR=%.2e",
            epoch + 1, max_epochs,
            history['train_loss'][-1], history['train_acc'][-1],
            history['val_loss'][-1], history['val_acc'][-1],
            history['test_loss'][-1], history['test_acc'][-1],
            history['lr'][-1],
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_without_improvement = 0
            # deepcopy required: state_dict() returns references, not copies
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            logger.info("Early stopping at epoch %d (patience=%d)", epoch + 1, patience)
            break

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        logger.info("Restored best model (val_loss=%.4f)", best_val_loss)

    return model, history

## 7. Train RF-UAV-Net

In [9]:
num_classes = len(drone_classes)
rfuavnet = RFUAVNet(num_classes=num_classes)

logger.info("RF-UAVNet architecture:\n%s", rfuavnet)
logger.info("Parameters: %d", sum(p.numel() for p in rfuavnet.parameters()))

In [10]:
rfuavnet, history = train_model(rfuavnet, train_loader, val_loader, test_loader, CONFIG)
logger.info("Training complete (%d epochs)", len(history['train_loss']))

Epoch 61/120: 100%|██████████| 55/55 [00:03<00:00, 15.79it/s]


## 8. Plot Training History

In [11]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('RF-UAVNet Loss', 'RF-UAVNet Accuracy'))

epochs = list(range(1, len(history['train_loss']) + 1))

fig.add_trace(go.Scatter(x=epochs, y=history['train_loss'], mode='lines+markers',
                         name='Train Loss', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history['val_loss'], mode='lines+markers',
                         name='Val Loss', line=dict(color='orange')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history['test_loss'], mode='lines+markers',
                         name='Test Loss', line=dict(color='red', dash='dash')), row=1, col=1)

fig.add_trace(go.Scatter(x=epochs, y=history['train_acc'], mode='lines+markers',
                         name='Train Acc', line=dict(color='blue')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=history['val_acc'], mode='lines+markers',
                         name='Val Acc', line=dict(color='orange')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=history['test_acc'], mode='lines+markers',
                         name='Test Acc', line=dict(color='red', dash='dash')), row=1, col=2)

fig.update_layout(title='RF-UAVNet Training History', height=500, width=1200)
fig.update_xaxes(title_text='Epoch', row=1, col=1)
fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_xaxes(title_text='Epoch', row=1, col=2)
fig.update_yaxes(title_text='Accuracy', row=1, col=2)

fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 9. Evaluate on Test Set

In [12]:
def evaluate_model(model, data_loader, device):
    """Evaluate model and return predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_x, batch_y in data_loader:
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.numpy())

    return np.array(all_preds), np.array(all_labels)


preds, labels = evaluate_model(rfuavnet, test_loader, CONFIG['device'])
accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average='weighted')

present_labels = sorted(set(labels) | set(preds))
present_names = [drone_classes[i] for i in present_labels]

print(f"RF-UAVNet Test Accuracy: {accuracy:.4f}")
print(f"RF-UAVNet Test F1-Score (weighted): {f1:.4f}")
print("Classification Report:")
print(classification_report(labels, preds, labels=present_labels, target_names=present_names))

RF-UAVNet Test Accuracy: 0.6298
RF-UAVNet Test F1-Score (weighted): 0.6258
Classification Report:
              precision    recall  f1-score   support

         AIR       0.57      0.63      0.60       922
         DIS       0.89      0.84      0.87       600
         INS       0.62      0.73      0.67       800
         MA1       0.42      0.37      0.39      1000
         MAV       0.48      0.51      0.49       800
         MIN       0.94      0.97      0.96       800
         PHA       0.60      0.44      0.51       700

    accuracy                           0.63      5622
   macro avg       0.65      0.64      0.64      5622
weighted avg       0.63      0.63      0.63      5622



## 10. Confusion Matrix

In [13]:
cm = confusion_matrix(labels, preds)

# Create confusion matrix heatmap with plotly
fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=list(drone_classes),
    y=list(drone_classes),
    colorscale='Purples',
    text=cm,
    texttemplate='%{text}',
    textfont={'size': 12},
    hoverongaps=False
))

fig.update_layout(
    title=f'RF-UAVNet Confusion Matrix - Accuracy: {accuracy:.4f}',
    xaxis_title='Predicted',
    yaxis_title='True',
    xaxis={'side': 'bottom'},
    yaxis={'autorange': 'reversed'},
    width=800,
    height=700
)
fig.show()
save_figure(fig)

Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## 11. Per-Class Performance

In [14]:
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    labels, preds, labels=range(len(drone_classes)), zero_division=0
)

metrics_df = pd.DataFrame({
    'Class': drone_classes,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1_per_class,
    'Support': support,
})

print("\nPer-Class Performance:")
print(metrics_df.to_string(index=False))

fig_precision = px.bar(metrics_df, x='Class', y='Precision', title='RF-UAVNet Precision per Class',
                       color='Precision', range_y=[0, 1.05])
fig_precision.update_layout(xaxis_title="Class", yaxis_title="Precision", height=400)
fig_precision.show()
save_figure(fig_precision)

fig_recall = px.bar(metrics_df, x='Class', y='Recall', title='RF-UAVNet Recall per Class',
                    color='Recall', color_continuous_scale=px.colors.sequential.Oranges, range_y=[0, 1.05])
fig_recall.update_layout(xaxis_title="Class", yaxis_title="Recall", height=400)
fig_recall.show()
save_figure(fig_recall)

fig_f1 = px.bar(metrics_df, x='Class', y='F1-Score', title='RF-UAVNet F1-Score per Class',
                color='F1-Score', color_continuous_scale=px.colors.sequential.Greens, range_y=[0, 1.05])
fig_f1.update_layout(xaxis_title="Class", yaxis_title="F1-Score", height=400)
fig_f1.show()
save_figure(fig_f1)


Per-Class Performance:
Class  Precision   Recall  F1-Score  Support
  AIR   0.569756 0.633406  0.599897      922
  DIS   0.893993 0.843333  0.867925      600
  INS   0.616597 0.733750  0.670091      800
  MA1   0.418448 0.372000  0.393859     1000
  MAV   0.478312 0.510000  0.493648      800
  MIN   0.941889 0.972500  0.956950      800
  PHA   0.598826 0.437143  0.505367      700


Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido



## Critical Analysis

RF-UAVNet achieves 62.98% accuracy on 7 drone classes (vs 14.3% random baseline), stopped early at epoch 41. The model extracts meaningful discriminative patterns from raw IQ signals -- MIN and DIS are classified reliably (F1 > 0.85), while MA1 and PHA remain poorly separated (F1 < 0.50), indicating that the temporal features distinguishing these drones fall outside the bandwidth preserved after downsampling.

The dominant performance bottleneck is the 120x downsampling ratio (1.2M to 10K samples per segment). This temporal decimation irreversibly discards high-frequency content -- modulation transitions, frequency-hopping edges, burst boundaries -- that encodes drone-specific signatures. The model's 4,615 parameters are not the limiting factor; the input signal simply lacks the spectral resolution needed to separate all 7 classes.

The uneven per-class performance confirms this interpretation: drones with distinctive low-frequency envelopes (MIN, DIS) are well captured, while drones whose signatures differ primarily in high-frequency modulation details (MA1, PHA, MAV) collapse into confusion clusters.

## Conclusion

RF-UAVNet achieves **62.98% test accuracy** (weighted F1: 0.6258) on 7-class drone classification from raw IQ signals, trained with Adam (lr=1e-3, weight_decay=1e-4) and early stopping at epoch 41. The 120x downsampling ratio (1.2M to 10K samples) remains the dominant performance bottleneck, irreversibly discarding high-frequency modulation details that separate visually similar drones (MA1, PHA, MAV cluster with F1 < 0.51). Drones with distinctive low-frequency envelopes (MIN F1=0.95, DIS F1=0.85) are well captured despite the decimation.

The trained model is saved to `data/models/rfuavnet_iq.pth` for cross-model comparison in NB 031.

## 12. Save Model

In [15]:
models_dir = Path(CONFIG['models_dir'])
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / 'rfuavnet_iq.pth'

# Best model already restored by early stopping before this point
torch.save({
    'model_state_dict': rfuavnet.state_dict(),
    'history': history,
    'num_classes': num_classes,
    'drone_classes': drone_classes,
}, model_path)

logger.info("Model saved to %s", model_path)

## 13. Upload Model to Scaleway

In [16]:
if COLAB:
    import glob
    model_dir = CONFIG["models_dir"] if "CONFIG" in dir() else "../data/models"
    for model_file in sorted(glob.glob(os.path.join(model_dir, "rfuavnet*.pth"))):
        model_name = os.path.basename(model_file)
        s3.upload_file(model_file, ARTIFACT_BUCKET, f"models/{model_name}")
        print(f"Uploaded {model_name}")

Uploaded rfuavnet_iq.pth


## 14. Summary

In [17]:
n_total = len(X_train_t) + len(X_val_t) + len(X_test_t)

print("=" * 60)
print("RF-UAVNET TRAINING SUMMARY")
print("=" * 60)
print(f"\nDataset: {n_total} total | {len(X_train_t)} train | {len(X_val_t)} val | {len(X_test_t)} test | {num_classes} classes")
print(f"Training: max_epochs={CONFIG['max_epochs']}, patience={CONFIG['patience']}, batch_size={CONFIG['batch_size']}")
print(f"  Optimizer: Adam, LR: {CONFIG['learning_rate']}")
print(f"  Stopped at epoch {len(history['train_loss'])}")
print("\nTest set metrics:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  F1-Score (weighted): {f1:.4f}")
print(f"\nModel saved to: {model_path}")
print("=" * 60)

RF-UAVNET TRAINING SUMMARY

Dataset: 39000 total | 27778 train | 5600 val | 5622 test | 7 classes
Training: max_epochs=120, patience=10, batch_size=512
  Optimizer: Adam, LR: 0.001
  Stopped at epoch 61

Test set metrics:
  Accuracy: 0.6298
  F1-Score (weighted): 0.6258

Model saved to: ../data/models/rfuavnet_iq.pth
